# 05 · Flood Risk Scoring Engine (FRS)

**Step 5** — the platform's signature operational metric: the **Flood Risk Score (0–100)**.

The FRS blends the transparent weighted-indicator model with the ML flood-probability surface, responds dynamically to rainfall, and rolls up to per-zone scores, vulnerability categories and affected-zone counts.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np
from src import risk_scoring, modeling, utils
from src.feature_engineering import INDICATOR_NAMES
from src.utils import PROCESSED_DIR

indicators = {n: np.load(PROCESSED_DIR / f'indicator_{n}.npy') for n in INDICATOR_NAMES}
surface = np.load(PROCESSED_DIR / 'model_surface.npy') if (PROCESSED_DIR / 'model_surface.npy').exists() else None
scorer = risk_scoring.build_scorer_from_artifacts(indicators, model_surface=surface, model_blend=0.35)

In [ ]:
result = scorer.compute(rainfall_mm=utils.RAINFALL_REFERENCE_MM)
print('==== SIGNATURE METRIC ====')
for k, v in result.headline().items():
    print(f'{k:>22}: {v}')

## Dynamic response to rainfall

In [ ]:
for mm in [20, 60, 100, 150, 200]:
    r = scorer.compute(rainfall_mm=mm)
    print(f'Rainfall {mm:3d} mm -> FRS {r.mean_frs:5.1f} ({r.category:8s}) | affected zones: {r.affected_zones}')

In [ ]:
from src import visualization as viz
viz.rainfall_response_figure(scorer.rainfall_response_curve()).show()
viz.zone_bar_figure(result.zone_table).show()
result.zone_table

In [ ]:
np.save(PROCESSED_DIR / 'frs_surface.npy', result.frs_surface)
result.zone_table.to_csv(PROCESSED_DIR / 'zone_scores.csv', index=False)
result.hotspots.to_csv(PROCESSED_DIR / 'hotspots.csv', index=False)
print('Saved FRS surface, zone scores, hotspots. Next: 06_dashboard_visualization.ipynb')